# Reproducing the main results

This notebook checks that the code of this repository produces the numbers in the README. It does
**not** retrain anything: it downloads the LoRA adapters we published on Hugging Face, evaluates
them again on the test set from scratch (no cached results are used), and compares the new
predictions with the ones we saved during our runs.

- **Hugging Face token: optional, but recommended.** All datasets, adapters and result tables of
  the [bdatm-project](https://huggingface.co/bdatm-project) organization are public, and this
  notebook never writes to the Hub, so no access to our repositories is needed. But each dataset is
  stored as thousands of image files, and Hugging Face allows far fewer downloads without a token:
  without one, the downloads can stop with a rate-limit error. Any read-only token of a free
  Hugging Face account works (create one at
  [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)). On Colab, add it as a
  secret called `HF_TOKEN` (key icon in the left sidebar) and allow this notebook to access it:
  `huggingface_hub` reads it by itself, and the token never appears in the notebook. Locally, set
  the `HF_TOKEN` environment variable or run `hf auth login`.
- **GPU:** an NVIDIA L4 or A100 (on Colab: *Runtime > Change runtime type*). The model runs in
  bfloat16, which the free T4 GPU does not support well.
- **The predictions are not exactly the same as ours.** Decoding is greedy (no sampling), but GPU
  computations in bfloat16 have some numerical noise, and when two tokens are almost equally likely
  it is enough to pick the other one. This happens where the model is unsure: in Task 1 mostly in
  the parts of the FEN that cannot be seen in the image (side to move, move counters), and in the
  vanilla models almost everywhere. The scores should match ours or differ by one or two samples.
  A different GPU type can change a few more predictions. The README explains this in more detail.
- **torch version.** The predictions also depend on the torch version. All our results were
  produced with torch 2.11.0 (the CUDA 13.0 build preinstalled on Colab), and this notebook installs
  that version if Colab has a different one.

The `LEVEL` setting in Section 2 chooses how much to run. The outputs saved in this notebook come
from `LEVEL = "tables"`; to check the code quickly, choose `"main"` before running:

| `LEVEL` | What runs | Time on an L4 |
|---|---|---|
| `"main"` (quick check) | The raster LoRA model of each task, on all 400 test samples | about 35 min |
| `"tables"` (the saved outputs) | Every row of the README tables, including the 512px resolution ablation. Task 2 and Task 3 on all 400 samples; Task 1 on all 400 samples for the raster LoRA model and on the first 100 for the other rows | about 2.5 h |
| `"full"` | Every row of the README tables, including the 512px resolution ablation, on all 400 test samples | about 6.5 h |

These times are for the evaluation. Downloading the three datasets without a token takes about
40 minutes more in total (about 10 minutes each for Task 1 and Task 2, 20 for Task 3, measured on
our connection).

Task 1 is the slow one: a FEN answer is about 40 tokens long and is generated one token at a time.
Using 100 samples is still a fair check, because the comparison is done sample by sample against
our saved predictions for those same samples.

## 1. Setup

On Colab this cell clones the repository and installs the pinned packages of `requirements.txt`
except `torch` and `torchvision`. It keeps Colab's preinstalled torch if it is 2.11.0, the version
of our runs, and installs 2.11.0 otherwise. It also removes Colab's `torchao`, which is too old for
`peft`, and `torchaudio`, which is not used and would not match torch 2.11.0 if it has to be
installed. To run locally, set `CONFIG["colab"] = False`, install the packages with
`pip install -r requirements.txt` (it pins torch 2.11.0) and start Jupyter from the `notebooks/`
folder.

The next cell imports the libraries, checks the GPU and says whether a Hugging Face token was found
(see the top of the notebook).

In [1]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}
# torch version of our runs (see TORCH_VERSIONS in src/eval/reproduce.py)
TORCH, TORCHVISION = "2.11.0", "0.26.0"

if CONFIG["colab"]:
    repo_root = Path(CONFIG["repo_dir"])
    if not repo_root.exists():
        repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"
        print(f"Cloning {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True,
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
    # torch and torchvision are handled below: install everything else at the pinned versions
    requirements = [line for line in (repo_root / "requirements.txt").read_text().splitlines()
                    if line.strip() and not line.startswith(("#", "torch"))]
    Path("/tmp/requirements_colab.txt").write_text("\n".join(requirements))
    !pip install -q -r /tmp/requirements_colab.txt
    # Colab's preinstalled torchao is too old for peft, which then fails to load the adapters;
    # the task notebooks remove it too. torchaudio is not used and would not match torch 2.11.0.
    !pip uninstall -y -q torchao torchaudio
    # Our runs used Colab's torch 2.11.0 (CUDA 13.0 build)
    if importlib.metadata.version("torch").split("+")[0] != TORCH:
        !pip install -q torch=={TORCH} torchvision=={TORCHVISION} --index-url https://download.pytorch.org/whl/cu130
else:
    repo_root = Path(".").resolve().parent

print(f"Repository: {repo_root}")

Cloning https://github.com/Aivon99/BigDataAndTextMiningProject.git...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 94.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━

In [2]:
import pandas as pd
import torch
import transformers
import peft

sys.path.insert(0, str(repo_root / "src"))

from eval.reproduce import IMAGE_SIZE, check_token_reordering, make_processors, reproduce_task

print(f"torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}")
assert torch.cuda.is_available(), "This notebook needs a GPU (an L4 or A100)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
if not torch.cuda.is_bf16_supported():
    print("WARNING: this GPU does not support bfloat16 well; results may differ from ours.")

# Optional token: huggingface_hub reads it from the Colab secret HF_TOKEN (or the HF_TOKEN
# environment variable) by itself; without it the Hub allows far fewer downloads
from huggingface_hub import get_token
if get_token():
    print("Hugging Face token found: downloads use the higher rate limit.")
else:
    print("No Hugging Face token: downloading without one (the Hub may rate-limit the downloads).")

torch 2.11.0+cu130, transformers 5.20.2, peft 0.21.2
GPU: NVIDIA L4


## 2. Settings

`LEVEL` chooses which rows to reproduce (see the table at the top). On Colab it is a drop-down menu
next to the code; elsewhere, edit the value. The outputs saved in this notebook come from
`LEVEL = "tables"`; for a quick check choose `"main"`.

`NUM_WORKERS` sets how many processes download and load the datasets in parallel. It only changes
the loading time, not the results; a split stored as a single file is loaded by one process
whatever its value. The other settings (model, image sizes, scan orders, number of shots) are the
ones used in the task notebooks and are defined in `src/eval/reproduce.py`.

In [3]:
LEVEL = "tables"  # @param ["main", "tables", "full"]
TASK1_SUBSET = 100      # with LEVEL = "tables": test samples for the Task 1 rows other than raster LoRA
NUM_WORKERS = 8  # @param {type:"integer"}

assert LEVEL in ("main", "tables", "full"), f"Unknown LEVEL: {LEVEL}"

## 3. The evaluation code

The code that evaluates the models is in `src/eval/reproduce.py`, and the cells below call it.
Importing it turns off the progress bars of the downloads, which would print one bar for each of
the thousands of image files of a dataset; the evaluation keeps its own progress bar.

- `load_task_dataset` loads a task's dataset from the Hub the same way the task notebooks do.
  Task 3 has two images per sample, so its files are downloaded first and the second frame
  (`image_t1`) is added to each row.
- `experiment_rows` lists the rows of the README tables for a task: the model name used in the
  results table, the adapter on the Hub (`None` for the vanilla model), the scan order used at
  test time and the image size. For the learned orders, the order is read from
  `learned_order.json` in the adapter's repository. The last row is the 512px raster LoRA model of
  the resolution ablation (repository `qwen-taskN-standard-lora`, named
  `Qwen + LoRA Fine-Tuning (from HF)` in the results table).
- `evaluate_row` loads the base model and the adapter, sets the scan order and runs the task's
  evaluation function of `src/eval/evaluation.py`, with the processor of the row's image size.
- `compare_with_published` downloads our saved per-sample predictions for the same model and
  compares them with the new ones, sample by sample.
- `reproduce_task` runs the rows selected by `LEVEL` for one task and returns the comparison. It
  first prints a warning if the installed torch is not the one of our runs of that task.

## 4. Processors and token reordering check

Two processors are prepared: one resizes the boards to 256px (all the main experiments) and one to
512px (the resolution ablation). Each prints the token grid it gives: 8x8 tokens at 256px (one per
square) and 16x16 at 512px (2x2 per square).

The scan orders are applied by patching the model's code at run time
(`src/training/token_reordering.py`). `verify_token_reordering` checks that the patch works with
the installed `transformers` version: with no order set the output equals the unpatched model,
the token embeddings and their positions move together, and setting an order changes the output.

In [4]:
processors = make_processors()
check_token_reordering(processors[IMAGE_SIZE])

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[token reordering] OK: {'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.03125}


## 5. Evaluate and compare

For each task: load the dataset and our published results table, then evaluate every selected row
and compare it with our saved predictions. Every row prints a progress bar; Task 1 rows take
about 20-25 minutes each on all 400 samples, Task 2 and Task 3 rows about 2 minutes. The 512px
rows are slower, because each image gives 4 times as many visual tokens.

In the comparison table:

- **ours, same samples**: the score of our saved predictions on the samples evaluated here;
- **reproduced**: the score of the new predictions on the same samples;
- **identical predictions**: how many new predictions are exactly the same string as ours;
- **ours, 400 samples (README)**: the number reported in the README.

### Task 1

In [5]:
comparison_task1 = reproduce_task("task1", LEVEL, processors, NUM_WORKERS, TASK1_SUBSET)
display(pd.DataFrame(comparison_task1))

Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.9kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Evaluating Qwen + LoRA Fine-Tuning (from HF): 100%|██████████| 100/100 [05:16<00:00,  3.16s/it]



Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task1_qwen_+_lora_fine-tuning_(from_hf)_results.csv.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/87.9k [00:00<?, ?B/s]

{'task': 'task1', 'model': 'Qwen + LoRA (Raster - SFT) [512px]', 'test samples': 100, 'ours, same samples (%)': 100.0, 'reproduced (%)': 99.0, 'identical predictions': '82/100', 'ours, 400 samples (README, %)': 99.0}


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)"
0,task1,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],100,0.0,0.0,86/100,0.00
1,task1,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],100,0.0,0.0,78/100,0.00
2,task1,Qwen + LoRA (Raster - SFT) [tok256],400,97.0,97.0,338/400,97.00
3,task1,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",100,0.0,0.0,60/100,0.00
4,task1,"Qwen + LoRA Raster, read spiral_center_out (TF...",100,0.0,0.0,57/100,0.00
5,task1,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",100,0.0,0.0,49/100,0.00
6,task1,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",100,0.0,0.0,57/100,0.00
7,task1,Qwen + LoRA (Zigzag - SFT) [tok256],100,97.0,97.0,94/100,97.50
8,task1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],100,99.0,99.0,83/100,98.50
9,task1,Qwen + LoRA (File_wise - SFT) [tok256],100,95.0,95.0,82/100,96.75


### Task 2

In [7]:
comparison_task2 = reproduce_task("task2", LEVEL, processors, NUM_WORKERS, TASK1_SUBSET)
display(pd.DataFrame(comparison_task2))

Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

metadata.jsonl:   0%|          | 0.00/213k [00:00<?, ?B/s]

Setting num_proc from 8 back to 1 for the train split to disable multiprocessing as it only contains one shard.


...

Evaluating Qwen + LoRA Fine-Tuning (from HF) (Task 2): 100%|██████████| 400/400 [02:03<00:00,  3.25it/s]



Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task2_qwen_+_lora_fine-tuning_(from_hf)_results.csv.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

{'task': 'task2', 'model': 'Qwen + LoRA (Raster - SFT) [512px]', 'test samples': 400, 'ours, same samples (%)': 62.0, 'reproduced (%)': 62.5, 'identical predictions': '396/400', 'ours, 400 samples (README, %)': 62.0}


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)"
0,task2,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],400,0.00,0.25,252/400,0.00
1,task2,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],400,0.75,0.75,317/400,0.75
2,task2,Qwen + LoRA (Raster - SFT) [tok256],400,60.25,60.00,399/400,60.25
3,task2,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",400,52.75,52.25,393/400,52.75
4,task2,"Qwen + LoRA Raster, read spiral_center_out (TF...",400,14.50,14.00,391/400,14.50
5,task2,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",400,9.75,9.50,391/400,9.75
6,task2,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",400,29.00,29.50,388/400,29.00
7,task2,Qwen + LoRA (Zigzag - SFT) [tok256],400,53.50,53.50,400/400,53.50
8,task2,Qwen + LoRA (Spiral_center_out - SFT) [tok256],400,54.00,54.00,400/400,54.00
9,task2,Qwen + LoRA (File_wise - SFT) [tok256],400,54.75,54.75,400/400,54.75


### Task 3

In [8]:
comparison_task3 = reproduce_task("task3", LEVEL, processors, NUM_WORKERS, TASK1_SUBSET)
display(pd.DataFrame(comparison_task3))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8004 files:   0%|          | 0/8004 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/6401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

...

Evaluating Qwen + LoRA Fine-Tuning (from HF) on Task 3: 100%|██████████| 400/400 [02:07<00:00,  3.14it/s]



Evaluation completed for Qwen + LoRA Fine-Tuning (from HF) on Task 3! Results saved to task3_qwen_+_lora_fine-tuning_(from_hf)_results.csv.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

{'task': 'task3', 'model': 'Qwen + LoRA (Raster - SFT) [512px]', 'test samples': 400, 'ours, same samples (%)': 87.5, 'reproduced (%)': 87.5, 'identical predictions': '400/400', 'ours, 400 samples (README, %)': 87.5}


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)"
0,task3,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],400,0.25,0.00,322/400,0.25
1,task3,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],400,0.00,0.00,399/400,0.00
2,task3,Qwen + LoRA (Raster - SFT) [tok256],400,79.75,80.25,398/400,79.75
3,task3,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",400,42.25,42.00,396/400,42.25
4,task3,"Qwen + LoRA Raster, read spiral_center_out (TF...",400,5.75,6.25,392/400,5.75
5,task3,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",400,4.00,4.25,387/400,4.00
6,task3,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",400,12.75,12.75,393/400,12.75
7,task3,Qwen + LoRA (Zigzag - SFT) [tok256],400,77.50,77.50,396/400,77.50
8,task3,Qwen + LoRA (Spiral_center_out - SFT) [tok256],400,70.75,71.00,395/400,70.75
9,task3,Qwen + LoRA (File_wise - SFT) [tok256],400,80.50,80.50,400/400,80.50


## 6. Summary

All the reproduced rows together. The scores should match ours or differ by one or two samples.
Some predictions can differ because of the numerical noise of the GPU, mostly where the model is
unsure (see the top of the notebook).

In [9]:
summary = pd.DataFrame(comparison_task1 + comparison_task2 + comparison_task3)
summary["difference (points)"] = (summary["reproduced (%)"] - summary["ours, same samples (%)"]).round(2)
print(f"LEVEL = {LEVEL!r}, GPU: {torch.cuda.get_device_name(0)}")
display(summary)

LEVEL = 'tables', GPU: NVIDIA L4


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)",difference (points)
0,task1,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],100,0.00,0.00,86/100,0.00,0.00
1,task1,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],100,0.00,0.00,78/100,0.00,0.00
2,task1,Qwen + LoRA (Raster - SFT) [tok256],400,97.00,97.00,338/400,97.00,0.00
3,task1,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",100,0.00,0.00,60/100,0.00,0.00
4,task1,"Qwen + LoRA Raster, read spiral_center_out (TF...",100,0.00,0.00,57/100,0.00,0.00
5,task1,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",100,0.00,0.00,49/100,0.00,0.00
6,task1,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",100,0.00,0.00,57/100,0.00,0.00
7,task1,Qwen + LoRA (Zigzag - SFT) [tok256],100,97.00,97.00,94/100,97.50,0.00
8,task1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],100,99.00,99.00,83/100,98.50,0.00
9,task1,Qwen + LoRA (File_wise - SFT) [tok256],100,95.00,95.00,82/100,96.75,0.00
